## 1. Install required Python packages

In [1]:
%pip install langchain langgraph langchain-openai

Note: you may need to restart the kernel to use updated packages.


## 2. Get Endpoints

Retrieve the FQDN of the self-hosted LLM and the Cosmos DB connection details from Terraform outputs.

In [2]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

aca_qwen_36_35b_a100_fqdn = ! terraform -chdir=infra output -raw aca_qwen_36_35b_a100_fqdn
aca_qwen_36_35b_a100_fqdn = aca_qwen_36_35b_a100_fqdn.n
print("LLM Endpoint:", aca_qwen_36_35b_a100_fqdn)

foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

LLM Endpoint: ╷
│ Error: Output "aca_gemma4_31b_it_a100_fqdn" not found
│ 
│ The output variable requested could not be found in the state file. If you
│ recently added this to your configuration, be sure to run `terraform
│ apply`, since the state won't be updated with new output variables until
│ that command is run.
╵
LLM Endpoint: qwen-3-6-35b-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4


## 3. Set Up the LLM Model

Create a `ChatOpenAI` model pointing at the vLLM-compatible endpoint running on Azure Container Apps.

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    base_url=f"http://{aca_qwen_36_35b_a100_fqdn}/v1",
    api_key="EMPTY",
    model=" @$",
    streaming=True,
    max_completion_tokens=512
)

# model = ChatOpenAI(
#     base_url=f"{foundry_endpoint}/openai/v1",
#     api_key=foundry_api_key,
#     model=llm_model_deployment_name_chatgpt,
#     streaming=True,
#     max_completion_tokens=512
# )

## 4. Test the Model

Invoke the model with a test prompt to ensure it's working correctly.

In [4]:
from langchain_core.messages import HumanMessage

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)



I'm Qwen, a large language model developed by Alibaba Group's Tongyi Lab. I'm designed to be a helpful, versatile AI partner—ready to assist with answering questions, analyzing information, writing, coding, brainstorming, and tackling complex reasoning tasks. I support a wide range of languages and aim to communicate clearly, accurately, and in a way that aligns with your goals. 

How can I help you today?

## 5. Use the Model in a LangChain Agent

In [5]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    middleware=[],
    checkpointer=None,
)

response = agent.stream({"messages": HumanMessage(content="Tell me about yourself")})

async for step in agent.astream(
    {"messages": [HumanMessage(content="Tell me about yourself")]},
    stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me about yourself
================================== Ai Message ==================================



I'm Qwen, a large language model independently developed by Alibaba Group's Tongyi Lab. I'm designed to be a clear, helpful, and capable thinking partner. I can assist with a wide range of tasks, including understanding and generating text in over 100 languages, complex reasoning, coding, data analysis, creative writing, and structured problem-solving. My goal is to adapt to your needs, provide accurate and thoughtful responses, and make collaboration as seamless as possible. 

How can I help you today?


## 6. Human-in-the-Loop Middleware

Add human oversight to tool calls using `HumanInTheLoopMiddleware`. When the model
proposes a tool call that matches the configured policy, the middleware fires an
`interrupt` that pauses execution. A reviewer can then **approve**, **edit**,
**reject**, or **respond** to the proposed action before the agent continues.

Reference: https://docs.langchain.com/oss/python/langchain/human-in-the-loop

### 6.1 Define some tools

We define three tools with different risk profiles:

- `write_file` — sensitive (all decisions allowed)
- `execute_sql` — sensitive but no edits (approve / reject only)
- `read_data` — safe (no approval needed)


In [6]:
from langchain_core.tools import tool


@tool
def write_file(path: str, content: str) -> str:
    """Write `content` to the file at `path`. Sensitive: requires approval."""
    # In a real app this would touch the filesystem.
    return f"Wrote {len(content)} chars to {path}"


@tool
def execute_sql(query: str) -> str:
    """Execute a SQL statement. Sensitive: requires approval (no edits)."""
    # In a real app this would run against a database.
    return f"Executed: {query}"


@tool
def read_data(table: str) -> str:
    """Read rows from a table. Safe: no approval needed."""
    return f"Read 3 rows from {table}"


### 6.2 Create an agent with `HumanInTheLoopMiddleware`

The `interrupt_on` mapping tells the middleware which tool calls should pause for review:

- `True` &rarr; all four decision types (`approve`, `edit`, `reject`, `respond`) are allowed
- `{"allowed_decisions": [...]}` &rarr; restrict to a subset of decisions
- `False` &rarr; never interrupt

HITL requires a checkpointer so the graph state can be saved across the pause.
For prototyping we use `InMemorySaver`; in production use a persistent
checkpointer such as `AsyncPostgresSaver`.


In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

hitl_agent = create_agent(
    model=model,
    tools=[write_file, execute_sql, read_data],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # All decisions allowed: approve, edit, reject, respond
                "write_file": True,
                # Only approve or reject (no editing of the SQL query)
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},
                # Safe read-only operation: never interrupt
                "read_data": False,
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    # HITL needs a checkpointer to persist state across interrupts.
    checkpointer=InMemorySaver(),
)

### 6.3 Run until interrupt

Invoke the agent with `version="v2"` and a `thread_id` in `config`. The thread
ID lets us resume the same conversation after the human review.

With `version="v2"`, the result is a `GraphOutput` exposing an `.interrupts`
attribute containing the actions waiting for a decision.


In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "hitl-demo-1"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Please run this SQL to clean up old data: "
                    "DELETE FROM records WHERE created_at < NOW() - INTERVAL '30 days';"
                ),
            }
        ]
    },
    config=config,
    version="v2",
)

# The agent paused -- inspect the actions awaiting review.
print(result.interrupts)

(Interrupt(value={'action_requests': [{'name': 'execute_sql', 'args': {'query': "DELETE FROM records WHERE created_at < NOW() - INTERVAL '30 days';"}, 'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {\'query\': "DELETE FROM records WHERE created_at < NOW() - INTERVAL \'30 days\';"}'}], 'review_configs': [{'action_name': 'execute_sql', 'allowed_decisions': ['approve', 'reject']}]}, id='62ad841416cd9bf68efb3d72afec435a'),)


### 6.4 Approve and resume

Resume the paused conversation by re-invoking the agent on the same
`thread_id` with a `Command(resume=...)` that contains one decision per
pending action, **in the same order** they appeared in the interrupt request.

For `execute_sql` we configured `["approve", "reject"]`, so let's approve it.


In [11]:
approved = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
    version="v2",
)

for msg in approved.value["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Please run this SQL to clean up old data: DELETE FROM records WHERE created_at < NOW() - INTERVAL '30 days';
================================== Ai Message ==================================
================================ Human Message =================================

Please run this SQL to clean up old data: DELETE FROM records WHERE created_at < NOW() - INTERVAL '30 days';
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_5ce459c224e8460594b6f50f)
 Call ID: call_5ce459c224e8460594b6f50f
  Args:
================================ Human Message =================================

Please run this SQL to clean up old data: DELETE FROM records WHERE created_at < NOW() - INTERVAL '30 days';
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_c7b8d40a8d1145159aa5f3f9)
 Call ID: call_c7b8d

### 6.5 Edit a tool call before it runs

`write_file` is configured with `True`, so every decision is allowed. Below we
trigger a `write_file` call, then resume with an `edit` decision that rewrites
the arguments before the tool actually executes.

We use a new `thread_id` so this run is independent of the previous one.


In [15]:
edit_config = {"configurable": {"thread_id": "hitl-demo-2"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Write the text 'hello world' to the file /tmp/note.txt",
            }
        ]
    },
    config=edit_config,
    version="v2",
)

# Show what the model proposed before we override it.
print(result.interrupts)

# Resume with an edited action: change the path and content.
edited = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "write_file",
                        "args": {
                            "path": "/tmp/note-edited.txt",
                            "content": "hello world (reviewed by human)",
                        },
                    },
                }
            ]
        }
    ),
    config=edit_config,
    version="v2",
)

for msg in edited.value["messages"]:
    msg.pretty_print()


(Interrupt(value={'action_requests': [{'name': 'write_file', 'args': {'path': '/tmp/note.txt', 'content': 'hello world'}, 'description': "Tool execution pending approval\n\nTool: write_file\nArgs: {'path': '/tmp/note.txt', 'content': 'hello world'}"}], 'review_configs': [{'action_name': 'write_file', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='452e0fa9603c5a4fbd0ef827c5718ba5'),)
================================ Human Message =================================

Write the text 'hello world' to the file /tmp/note.txt
================================== Ai Message ==================================
Tool Calls:
  write_file (call_9bacfca1ccac4591ba00fd75)
 Call ID: call_9bacfca1ccac4591ba00fd75
  Args:
    path: /tmp/note.txt
    content: hello world
================================ Human Message =================================

Write the text 'hello world' to the file /tmp/note.txt
================================== Ai Message ==================================
Tool Calls:
